# 문제 1-2 : 2단계 Multi Chain 만들기 — 여행지 정보 시스템

여행지(도시/국가)를 입력하면 **1단계**에서 대표 명소 1곳을 추천하고, **2단계**에서 그 명소의 상세 정보(역사·특징·방문 팁)를 알려준다.

**요구사항 체크**
- [x] 1단계 체인: 여행지 → 대표 명소 1가지
- [x] 2단계 체인: 명소 → 상세 정보
- [x] `ChatPromptTemplate` 에 `"system"` / `"user"` 메시지 지정
- [x] 두 체인을 LCEL로 연결
- [x] 각 단계의 결과를 모두 출력

## 1. 환경 변수 로드

In [ ]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정 (.env 는 프로젝트 루트에 있음)
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 키 전체는 출력하지 않고 설정 여부만 확인
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

## 2. LLM 설정 (ChatOpenAI)

In [ ]:
from langchain_openai import ChatOpenAI

# 사용할 LLM 제공자 선택: "groq"(무료) 또는 "openai" — 이 값만 바꾸면 아래 모든 셀의 모델이 바뀜
PROVIDER = "groq"

# Groq는 OpenAI 호환 API라서 ChatOpenAI에 base_url만 바꿔서 사용
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini",
    },
}


def get_llm(temperature: float = 0.7, provider: str | None = None) -> ChatOpenAI:
    """선택한 제공자의 ChatOpenAI 인스턴스를 생성합니다."""
    config = LLM_CONFIGS[provider or PROVIDER]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {(provider or PROVIDER).upper()}_API_KEY 가 없습니다.")
    return ChatOpenAI(**config, temperature=temperature, max_retries=3)


llm = get_llm()
print(f"{PROVIDER} / {llm.model_name}")

## 3. 1단계 체인 — 여행지 → 대표 명소 이름
2단계 입력으로 바로 쓰기 위해 **명소 이름만** 출력하도록 system 메시지에서 제한한다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt1 = ChatPromptTemplate.from_messages([
    ("system", "당신은 세계 여행 전문가입니다. "
               "사용자가 말한 여행지의 가장 대표적인 관광 명소 1곳을 추천합니다. "
               "설명 없이 명소 이름만 한국어로 한 줄로 답하세요. (예: 콜로세움)"),
    ("user", "{destination}에서 꼭 가봐야 할 대표 관광 명소 1곳을 추천해 주세요."),
])

# 이름만 받기 때문에 temperature 를 낮춰 답변을 안정적으로 만든다
chain1 = prompt1 | get_llm(temperature=0.2) | StrOutputParser()

attraction = chain1.invoke({"destination": "로마"})
print("1단계 결과(추천 명소):", attraction)

## 4. 2단계 체인 — 명소 → 상세 정보 (역사 / 특징 / 방문 팁)

In [ ]:
prompt2 = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 여행 가이드입니다. 관광 명소에 대해 정확하고 알기 쉽게 설명합니다."),
    ("user", """다음 관광 명소의 상세 정보를 아래 형식으로 한국어로 알려주세요.

명소: {attraction}

형식:
명소: (명소 이름)
역사:
(2~3문장)
특징:
(2~3문장)
방문 팁:
(2~3가지, '- '로 시작)"""),
])

chain2 = prompt2 | llm | StrOutputParser()

detail = chain2.invoke({"attraction": attraction})
print("2단계 결과(상세 정보):")
print(detail)

## 5. 두 체인을 LCEL로 연결

수업에서 배운 방식: `{"attraction": chain1}` 딕셔너리로 1단계 출력을 2단계 프롬프트 변수에 넣는다.
이 방식은 최종 결과(2단계)만 돌려준다.

In [ ]:
multi_chain = {"attraction": chain1} | chain2

print(multi_chain.invoke({"destination": "뉴욕"}))

## 6. 각 단계 결과를 모두 확인하는 연결 — `RunnablePassthrough.assign`

`assign`은 입력 딕셔너리를 그대로 두고 새 키를 추가한다. 그래서 체인 하나를 실행해도
`destination` → `attraction`(1단계) → `detail`(2단계)이 모두 결과에 남는다.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

full_chain = (
    RunnablePassthrough.assign(attraction=chain1)   # 1단계 결과를 attraction 키로 추가
    | RunnablePassthrough.assign(detail=chain2)     # 2단계 결과를 detail 키로 추가
)


def travel_info(destination: str) -> dict:
    result = full_chain.invoke({"destination": destination})
    print(f"입력: \"{result['destination']}\"")
    print(f"1단계 결과: \"{result['attraction'].strip()}\" 추천")
    print("2단계 결과:")
    print(result["detail"])
    print("=" * 60)
    return result


for city in ["로마", "뉴욕", "도쿄"]:
    travel_info(city)